# PROC NESTED를 이용한 보험사 조직 위계구조 전반의 보험금 정산 변동성 분해

## 요약

손해보험사는 청구 정산 결과의 불일치가 **어디서** 비롯되는지 알고 싶어 한다: 지리적 지역 간 차이, 지점 간 차이, 개별 손해사정사 간 차이 때문인가, 아니면 단순히 건별 잡음 때문인가? 이 노트북은 완전히 중첩된 자동차 보험금 청구 합성 데이터(지역 > 지점 > 손해사정사 > 청구건)를 생성하고, **PROC NESTED**를 사용해 무작위효과 분산분석을 수행하여 위계구조의 각 수준이 기여하는 분산성분을 추정하고 이를 전체 대비 백분율로 보고한다. 그 결과는 경영진에게 정산 일관성 개선을 위해 어느 조직 계층을 우선 목표로 삼아야 하는지 알려 준다.

## 데이터 출처

모든 데이터는 첫 번째 DATA step에서 인라인으로 생성된다 — 외부 파일도, 네트워크도 없다. 설계는 **완전히 중첩**되어 있다: 각 지점은 정확히 하나의 지역에 속하고, 각 손해사정사는 정확히 하나의 지점에 속하며, 각 청구건은 정확히 한 명의 손해사정사에게 속한다.

| 데이터셋 | 행 수 | 변수 | 유형 | 역할 | 설명 |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | char | CLASS (1수준, 최상위) | 지리적 지역 (5개 지역: 동북, 동남, 중부, 서부, 남서) |
| | | `Branch` | char | CLASS (2수준, Region 내 중첩) | 지점 코드 (지역당 지점 2개) |
| | | `Adjuster` | char | CLASS (3수준, Branch 내 중첩) | 손해사정사 ID (지점당 사정사 2명) |
| | | `Settlement` | num | VAR (종속변수) | 자동차 보험금 청구에 지급된 배상금(USD) |
| | | `CycleDays` | num | VAR (종속변수) | 사고 최초 접수일부터 정산까지의 일수 |

합성 구조: 지역 5개 x 지점 2개 x 사정사 2명 x 청구 2건 = 관측치 40개. `rand('NORMAL', ...)`를 통해 지역 무작위효과, 지역 내 지점 무작위효과, 지점 내 사정사 무작위효과, 청구 수준 잔차가 가법적으로 층을 이루어 분산성분을 복원할 수 있도록 한다. 지역효과는 가장 큰 표준편차(2,200)로 추출되어 청구가 *어디서* 처리되는지가 지배적 요인이 되도록 한다. `call streaminit(20260531)`로 시드를 고정한다. 간결하고 완전히 균형 잡힌 설계는 위계구조의 모든 수준이 실질적인 자유도를 갖도록 보장한다.

# PROC NESTED를 이용한 보험금 정산 변동성 분해

손해보험사가 자동차 보험금 청구 정산에 지급하는 금액을 검토하면 흔히 큰 편차를 발견하게 된다. 이러한 편차의 일부는 불가피하지만(모든 사고는 저마다 다르다), 일부는 **조직적** 요인을 반영한다 — 어떤 지역은 다른 지역보다 후하게 지급하고, 어떤 지점은 더 관대하며, 개별 손해사정사가 이상치일 수 있다.

데이터는 엄격하게 **중첩된**(위계적) 구조를 가진다:

```
지역(Region)  ->  지점(Branch, Region 내 중첩)  ->  손해사정사(Adjuster, Branch 내 중첩)  ->  개별 청구건
```

지점은 정확히 하나의 지역에 속하고 손해사정사는 정확히 하나의 지점에 속하므로, 이 요인들은 교차(crossed)가 아니라 중첩(nested) 관계에 있다. `PROC NESTED`는 바로 이러한 설계를 위한 무작위효과 분산분석을 수행하며, 각 수준에서 **분산성분**을 추정하여 전체 대비 백분율로 표현한다. 그 백분율 분해는 다음과 같은 경영상의 질문에 답한다: *정산을 더 일관되게 만들려면 조직의 어느 계층을 목표로 삼아야 하는가?*

## 1단계 &mdash; 완전히 중첩된 합성 청구 데이터 생성

지역 5개, 각 지역당 지점 2개, 각 지점당 사정사 2명, 각 사정사가 처리하는 청구 2건(총 40행)을 시뮬레이션한다. 각 수준이 실제로 분산에 기여하도록 가법적 무작위효과로부터 반응변수를 구성한다:

- a **지역** 효과 (가장 큰 산포 &mdash; 지역 간 차이가 가장 큼),
- **지역 내 지점** 효과,
- **지점 내 사정사** 효과,
- 그리고 **청구 수준 잔차** (사정사 내 잡음).

`call streaminit`으로 재현성을 위해 시드를 고정하며, 모든 효과는 `rand('NORMAL', mean, sd)`로 추출한다. 지역효과는 가장 넓은 표준편차를 사용하므로 전체 분산 중 가장 큰 몫을 차지할 것으로 예상된다. 이후 다중 반응변수 분석을 보이기 위해 동일한 위계구조를 공유하는 두 번째 반응변수 `CycleDays`도 함께 생성한다.

In [1]:
데이터 claims;
   호출 streaminit(20260531);
   길이 Region $12 Branch $24 Adjuster $32;

   /* 지역 수준 무작위효과: 지역 간 차이가 가장 크다. */
   반복 r = 1 까지 5;
      Region = SCAN('동북 동남 중부 서부 남서', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* 지역 내에 중첩된 지점. */
      반복 b = 1 까지 2;
         Branch = catx('-', Region, PUT(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* 지점 내에 중첩된 손해사정사. */
         반복 a = 1 까지 2;
            Adjuster = catx('-', Branch, PUT(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* 이 손해사정사가 처리하는 개별 청구건. */
            반복 claim = 1 까지 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               만약 Settlement < 0 이면 Settlement = 0;
               만약 CycleDays  < 1 이면 CycleDays  = 1;
               출력;
            종료;
         종료;
      종료;
   종료;

   유지 Region Branch Adjuster Settlement CycleDays;
실행;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## 2단계 &mdash; 중첩 위계구조 순으로 정렬

`PROC NESTED`는 입력 데이터가 **나열할 순서대로 CLASS 변수 기준으로 정렬**되어 있어야 한다 &mdash; 가장 바깥쪽 요인이 먼저 온다. 프로시저가 위계구조를 올바르게 따라갈 수 있도록 `Region Branch Adjuster` 순으로 정렬한다.

In [2]:
처리 정렬 데이터=claims;
   기준 Region Branch Adjuster;
실행;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## 3단계 &mdash; 정산금액에 대한 분산성분 분석

핵심 분석이다. CLASS 변수를 바깥쪽에서 안쪽 순으로(`Region Branch Adjuster`) 나열한다; 가장 안쪽의 반복 단위인 개별 청구건은 자동으로 오차항으로 처리된다. `VAR Settlement`로 단일 반응변수를 지정한다.

`LABEL` 문은 출력 배너에 표시될 반응변수의 더 읽기 쉬운 표시 이름을 제공한다. 출력에는 기대평균제곱의 계수, (각 분산성분에 대한 0 여부의 F 검정을 포함한) 분산분석표, 그리고 각 수준의 **분산성분** 추정치와 그 **전체 대비 백분율**이 산출된다.

In [3]:
제목 '자동차 보험금 정산의 분산성분';
title2 '지역/지점/손해사정사 무작위효과 분산분석';

처리 nested 데이터=claims;
   분류 Region Branch Adjuster;
   변수 Settlement;
   라벨 Settlement = '보험금 지급액(USD)';
실행;


                                                    자동차 보험금 정산의 분산성분                                                    
                                                 지역/지점/손해사정사 무작위효과 분산분석                                                 


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable 보험금 지급액(USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826901       


NOTE: Option TITLE changed to 자동차 보험금 정산의 분산성분.
NOTE: Option TITLE2 changed to 지역/지점/손해사정사 무작위효과 분산분석.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## 4단계 &mdash; 두 반응변수 동시 분석

경영진은 **처리기간**(청구 정산에 걸리는 시간)에도 관심이 있다. `VAR` 목록에 `CycleDays`를 추가한다. 종속변수가 둘 이상이면 `PROC NESTED`는 위계구조의 각 수준에서 두 반응변수가 어떻게 공변하는지 보여주는 **공변동 분석**을 추가로 보고한다(예: 지급액이 더 큰 지역이 정산도 더 느린지 여부).

In [4]:
제목 '정산금액과 처리기간의 분산성분';
title2 '동일한 위계구조에서의 두 반응변수 분석';

처리 nested 데이터=claims;
   분류 Region Branch Adjuster;
   변수 Settlement CycleDays;
   라벨 Settlement = '보험금 지급액(USD)'
         CycleDays  = '정산 소요일수';
실행;


                                                    정산금액과 처리기간의 분산성분                                                    
                                                 동일한 위계구조에서의 두 반응변수 분석                                                  


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable 보험금 지급액(USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826901       


NOTE: Option TITLE changed to 정산금액과 처리기간의 분산성분.
NOTE: Option TITLE2 changed to 동일한 위계구조에서의 두 반응변수 분석.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## 5단계 &mdash; AOV 옵션을 이용한 분산성분 전용 보기

두 반응변수에 대한 분산성분을 빠르게 요약하려면 `AOV` 옵션으로 출력을 분산분석 통계량으로만 제한하고 **공변동 분석 섹션을 생략**한다. 이는 포트폴리오 분석가가 반응변수별 수준별 분산 분해만 필요하고 반응변수 간 공변동은 필요 없을 때 훑어보는 간결한 화면이다.

In [5]:
제목 '분산성분만 표시 (AOV)';

처리 nested 데이터=claims aov;
   분류 Region Branch Adjuster;
   변수 Settlement CycleDays;
   라벨 Settlement = '보험금 지급액(USD)'
         CycleDays  = '정산 소요일수';
실행;

제목;


                                                     분산성분만 표시 (AOV)                                                     
                                                 동일한 위계구조에서의 두 반응변수 분석                                                  


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable 보험금 지급액(USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Region                 4  62114122.126866          6.71      0.0304    Branch  15528530.531717  1651824.844989             54.4057      8.0000
Branch                 5  11569658.859028          1.89      0.1829  Adjuster  2313931.771806   272682.828942              8.9813      4.0000
Adjuster              10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826901       


NOTE: Option TITLE changed to 분산성분만 표시 (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## 결과 해석

분산분석표의 **전체 대비 백분율(Percent of Total)** 열이 핵심이다. 위에서 아래로 읽으면 전체 정산금액 변동성 중 각 조직 계층이 차지하는 몫을 알 수 있다. 정산금액에 대해서는 다음과 같은 결과가 나온다:

- **지역(Region) &mdash; 54.4%** (Pr > F = 0.0304). 데이터는 지역 수준에서 가장 큰 산포로 생성되었고, 분석은 이를 정확히 복원한다: 전체 정산금액 변동성의 절반 이상이 지역 *간* 차이이며, 이는 청구가 *어디서* 처리되는지가 지배적 요인이라는 통계적으로 유의한 증거이다.
- **지점(지역 내) &mdash; 9.0%** (Pr > F = 0.18). 지역에서 개별 지점으로 내려갈 때 추가되는 소폭의 몫이며, 여기서는 유의하지 않다.
- **손해사정사(지점 내) &mdash; 3.7%** (Pr > F = 0.33). 이 사업 구성에서는 개별 손해사정사에게 귀속되는 변동이 거의 없다.
- **오차 &mdash; 32.9%.** 사정사 내에서 건별로 발생하는 잔차 잡음으로, 어떤 조직적 수단으로도 제거할 수 없는 불가피한 변동이다.

각 수준은 분산성분이 0이라는 귀무가설에 대한 **F 검정(Pr > F)**도 함께 제시한다. 지역의 작은 p-값(0.0304)은 표본추출에 의한 우연이 아니라 지역 간 실제 체계적 차이가 존재한다는 통계적 증거이다.

처리기간 반응변수도 비슷한 이야기를 들려준다: **지역이 정산 소요일수 변동의 45.8%**를 차지하며(Pr > F = 0.0448, 역시 유의함), 지점과 손해사정사는 한 자릿수의 몫을 차지하고 잔차가 40.1%를 차지한다. 따라서 보험사가 *얼마나 많이* 지급하는지와 *얼마나 오래* 걸리는지는 모두 무엇보다 먼저 지역에 의해 좌우된다.

두 반응변수 동시 분석은 **공변동 분석**을 추가한다. 여기서는 지역 수준이 교차곱을 주도하며, 전체 **상관계수는 -0.36**이다 &mdash; 이는 음의 관계로, 더 큰 금액을 지급하는 지역일수록 정산을 *더 빨리* 마무리하는 경향이 있음을 뜻하며 더 느린 것이 아니다. 이는 유용하고 비직관적인 발견이다 &mdash; 비용이 큰 지역이 느린 지역은 아니다.

**비즈니스 시사점:** 지역이 두 반응변수 모두에서 전체 대비 백분율 분해를 지배하므로, 보험사는 지점이나 손해사정사 수준의 개입에 투자하기 전에 먼저 *지역 간* 정산 지침과 승인 한도를 표준화해야 한다 &mdash; 그곳이 가장 크고 통계적으로 유의한 불일치가 존재하는 곳이다. 정산금액과 처리기간 간의 음의 상관관계는 가장 비싸면서 동시에 가장 느린 단일 지역은 없음을 의미하므로, 두 문제는 지역별로 별도의 프로그램으로 각각 해결할 수 있다. PROC NESTED는 "우리 정산이 일관성이 없다"는 막연한 느낌을 그 불일치가 어디에 있는지를 계층별로 실행 가능하게 규명하는 분석으로 바꾸어 준다.